# CNN on Synthetic Image Data

## Objective
Train a simple CNN on generated synthetic line-pattern images.

## Google Colab Steps
1. Run the setup cell.
2. Execute the main cell.
3. Save a copy to your Google Drive if needed.


In [ ]:
!pip -q install numpy pandas matplotlib seaborn scikit-learn tensorflow


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

def make_image(label):
    img = np.zeros((28, 28), dtype=np.float32)
    if label == 0:
        c = np.random.randint(6, 22)
        img[:, c - 1:c + 1] = 1.0
    else:
        r = np.random.randint(6, 22)
        img[r - 1:r + 1, :] = 1.0
    img += 0.12 * np.random.randn(28, 28).astype(np.float32)
    return np.clip(img, 0.0, 1.0)

n_samples = 1400
y = np.random.randint(0, 2, size=n_samples)
X = np.array([make_image(lbl) for lbl in y])[..., np.newaxis]

split = 1100
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(2, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=32, verbose=1)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", acc)

plt.figure(figsize=(8, 3))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_test[i, :, :, 0], cmap="gray")
    plt.title(f"Label: {y_test[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("CNN Accuracy Curves")
plt.legend()
plt.show()
